# Peak shaving vs TCIPC analysis

Let's check the results. First we need to copy the results from the cluster to
this PC:
```
rsync -avm --include='*/' --include='*.sql*' --exclude='*' jhummel@login.delftblue.tudelft.nl:../../scratch/jhummel/tip_clearance/data/optimal_tuning/ ./data/optimal_tuning/ --dry-run
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.interpolate import griddata
from weis.visualization.utils import load_OMsql

plt.style.use("journal.mplstyle")

%matplotlib widget

In [ ]:
# Define the logs to load and give them labels.
logs_to_load = {
    # "Free yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_free_yaw.sql",
    # "Zero yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_zero_yaw.sql",
    # "Free yaw": "../../../data/optimal_tuning/ps_vs_ipc_free_yaw/ps_vs_ipc_free_yaw_kw2.sql",
    # "Zero yaw": "../../../data/optimal_tuning/ps_vs_ipc_zero_yaw/ps_vs_ipc_zero_yaw_kw2.sql",
    "Free yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_free_yaw_hub.sql",
    "Zero yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_zero_yaw_hub.sql",
}

# Load all datasets.
all_data_dicts = {}
for log_name, log_fmt in logs_to_load.items():
    all_data_dicts[log_name] = load_OMsql(log_fmt)
    print(f"Loaded {log_name}: {all_data_dicts[log_name].keys()}")

In [ ]:
# Let's define how we load, scale, and label the data, then make a dataframe.
all_outputs = {
    # ROSCO variables.
    "TCIPC_MaxTipDeflection": {
        "key": "tune_rosco_ivc.TCIPC_MaxTipDeflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC reference (m)",
    },
    "TCIPC_TowerClearanceReference": {
        "key": "tune_rosco_ivc.TCIPC_MaxTipDeflection",
        "scaling": lambda x: 30 - x[0],
        "label": "TCIPC reference (m)",
    },
    "ps_percent": {
        "key": "tune_rosco_ivc.ps_percent",
        "scaling": lambda x: x[0],
        "label": "Peak shaving (-)",
    },
    "max_thrust_percent": {
        "key": "tune_rosco_ivc.ps_percent",
        "scaling": lambda x: 100 * x[0],
        "label": "Max thrust (%)",
    },
    "TCIPC_nHarmonics": {
        "key": "tune_rosco_ivc.TCIPC_nHarmonics",
        "scaling": lambda x: x[0],
        "label": "Number of harmonics",
    },
    "TCIPC_ZeroYawDeflection": {
        "key": "tune_rosco_ivc.TCIPC_ZeroYawDeflection",
        "scaling": lambda x: x[0],
        "label": "Zero yaw deflection",
    },
    "TCIPC_MaxPitchAmplitude": {
        "key": "tune_rosco_ivc.TCIPC_MaxPitchAmplitude",
        "scaling": lambda x: np.round(np.rad2deg(x[0]), 1),
        "label": "TCIPC max pitch amplitude (deg)",
    },
    # Objectives / responses.
    "aep": {
        "key": "aeroelastic.AEP",
        "scaling": lambda x: 1e-6 * x[0],
        "label": "AEP (GWh)",
    },
    "max_eff_TipDxc_towerPassing": {
        "key": "aeroelastic.max_eff_TipDxc_towerPassing",
        "scaling": lambda x: x[0],
        "label": "Max effective TipDxc tower passing (m)",
    },
    # "max_TipDxc_towerPassing": {
    #     "key": "aeroelastic.max_TipDxc_towerPassing",
    #     "scaling": lambda x: x[0],
    #     "label": "Max TipDxc tower passing (m)",
    # },
    "max_TipDxc_towerPassing_DLC": {
        "key": "aeroelastic.max_TipDxc_towerPassing_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max TipDxc tower passing DLC (-)",
    },
    "max_TipDxc_towerPassing_U": {
        "key": "aeroelastic.max_TipDxc_towerPassing_U",
        "scaling": lambda x: x[0],
        "label": "Max TipDxc tower passing U (m/s)",
    },
    # "tower_clearance": {
    #     "key": "aeroelastic.max_TipDxc_towerPassing",
    #     "scaling": lambda x: 30 - x[0],
    #     "label": "Tower clearance (m)",
    # },
    "eff_tower_clearance": {
        "key": "aeroelastic.max_eff_TipDxc_towerPassing",
        "scaling": lambda x: 30 - x[0],
        "label": "Tower clearance (m)",
    },
    # "TCIPC_amplitude_at_max_deflection": {
    #     "key": "aeroelastic.TCIPC_amplitude_at_max_deflection",
    #     "scaling": lambda x: x[0],
    #     "label": "TCIPC amplitude (deg)",
    # },
    # "TCIPC_amplitude_at_max_deflection_DLC": {
    #     "key": "aeroelastic.TCIPC_amplitude_at_max_deflection_DLC",
    #     "scaling": lambda x: str(x[0]),
    #     "label": "TCIPC amplitude at max deflection DLC (-)",
    # },
    # "TCIPC_amplitude_at_max_deflection_U": {
    #     "key": "aeroelastic.TCIPC_amplitude_at_max_deflection_U",
    #     "scaling": lambda x: x[0],
    #     "label": "TCIPC amplitude at max deflection U (m/s)",
    # },
    "TCIPC_max_pitch_amplitude": {
        "key": "aeroelastic.TCIPC_max_pitch_amplitude",
        "scaling": lambda x: x[0],
        "label": "TCIPC max pitch amplitude (deg)",
    },
    "TCIPC_max_pitch_amplitude_DLC": {
        "key": "aeroelastic.TCIPC_max_pitch_amplitude_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "TCIPC max pitch amplitude DLC (-)",
    },
    "TCIPC_max_pitch_amplitude_U": {
        "key": "aeroelastic.TCIPC_max_pitch_amplitude_U",
        "scaling": lambda x: x[0],
        "label": "TCIPC max pitch amplitude U (m/s)",
    },
    # Structural loads.
    "max_eff_TwrBsMyt": {
        "key": "aeroelastic.max_eff_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "Tower ultimate (MNm)",
    },
    # "max_TwrBsMyt": {
    #     "key": "aeroelastic.max_TwrBsMyt",
    #     "scaling": lambda x: x[0] / 1000,
    #     "label": "Max tower base Myt (MNm)",
    # },
    "max_TwrBsMyt_DLC": {
        "key": "aeroelastic.max_TwrBsMyt_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max tower base Myt DLC (-)",
    },
    "max_TwrBsMyt_U": {
        "key": "aeroelastic.max_TwrBsMyt_U",
        "scaling": lambda x: x[0],
        "label": "Max tower base Myt U (m/s)",
    },
    "DEL_TwrBsMyt": {
        "key": "aeroelastic.DEL_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "Tower fatigue (MNm)",
    },
    "damage_tower_base": {
        "key": "aeroelastic.damage_tower_base",
        "scaling": lambda x: x[0],
        "label": "Tower base damage (-)",
    },
    "max_eff_RootMyb": {
        "key": "aeroelastic.max_eff_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "Blade ultimate (MNm)",
    },
    # "max_RootMyb": {
    #     "key": "aeroelastic.max_RootMyb",
    #     "scaling": lambda x: x[0] / 1000,
    #     "label": "Max root Myb (MNm)",
    # },
    "max_RootMyb_DLC": {
        "key": "aeroelastic.max_RootMyb_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max root Myb DLC (-)",
    },
    "max_RootMyb_U": {
        "key": "aeroelastic.max_RootMyb_U",
        "scaling": lambda x: x[0],
        "label": "Max root Myb U (m/s)",
    },
    "DEL_RootMyb": {
        "key": "aeroelastic.DEL_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "Blade fatigue (MNm)",
    },
    "hub_Mxyz": {
        "key": "aeroelastic.hub_Mxyz",
        "scaling": lambda x: np.linalg.norm(x) / 1000,
        "label": "Hub moment magnitude (MNm)",
    },
    # Control activity.
    "max_pitch_rate_sim": {
        "key": "aeroelastic.max_pitch_rate_sim",
        "scaling": lambda x: x[0],
        "label": "Max pitch rate (deg/s)",
    },
    "max_pitch_rate_sim_DLC": {
        "key": "aeroelastic.max_pitch_rate_sim_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max pitch rate DLC (-)",
    },
    "max_pitch_rate_sim_U": {
        "key": "aeroelastic.max_pitch_rate_sim_U",
        "scaling": lambda x: x[0],
        "label": "Max pitch rate U (m/s)",
    },
    "avg_pitch_travel": {
        "key": "aeroelastic.avg_pitch_travel",
        "scaling": lambda x: x[0],
        "label": "Avg pitch travel (deg)",
    },
}

# Build dataframe from mapping for each log.
labels = {short: info["label"] for short, info in all_outputs.items()}
all_dfs = []

for log_name, data_dict in all_data_dicts.items():
    df_dict = {}
    for short_label, info in all_outputs.items():
        data = data_dict[info["key"]]
        scaled_data = list(map(info["scaling"], data))
        df_dict[short_label] = scaled_data

    df_temp = pd.DataFrame(df_dict)
    df_temp["log_name"] = log_name
    all_dfs.append(df_temp)

# Combine all dataframes.
df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined dataframe shape: {df.shape}")
df

In [ ]:
with pd.option_context(
    "display.max_rows",
    20,
    "display.max_columns",
    None,
    "display.precision",
    3,
):
    # display(df.sort_values("aep"))
    # display(
    #     df[df["max_thrust_percent"] == 76].sort_values(
    #         ["log_name", "TCIPC_TowerClearanceReference"]
    #     )
    # )
    # display(
    #     df[df["TCIPC_TowerClearanceReference"] == 30].sort_values(
    #         ["log_name", "TCIPC_TowerClearanceReference"]
    #     )
    # )
    # display(
    #     df[
    #         (df["log_name"] == "Free yaw") & (df["TCIPC_TowerClearanceReference"] == 0)
    #     ].sort_values("ps_percent")
    # )
    display(
        df[(df["log_name"] == "Free yaw") & (df["ps_percent"] == 1.0)].sort_values(
            "TCIPC_TowerClearanceReference"
        )
    )

In [ ]:
df = df[df["aep"] != 0.0].reset_index(drop=True)

In [ ]:
# Constants used throughout the notebook.
DESIGN_VARS = ("TCIPC_TowerClearanceReference", "max_thrust_percent")
REFERENCE_POINT = (5.0, 80)
BASELINE_LOG = "Baseline"

# # Extract four subsets from the full 3D dataset, each with a fixed
# # TCIPC_MaxPitchAmplitude, reducing to a 2D design space per case.
# case_filters = {
#     BASELINE_LOG: {
#         "source": "Free yaw",
#         "TCIPC_TowerClearanceReference": 30.0,
#         "TCIPC_ZeroYawDeflection": 0,
#     },
#     "Free yaw": {
#         "source": "Free yaw",
#         "TCIPC_ZeroYawDeflection": 0,
#     },
#     "Zero yaw": {
#         "source": "Zero yaw",
#         "TCIPC_ZeroYawDeflection": 1,
#     },
# }

# LOG_NAMES = list(case_filters.keys())

# filtered_dfs = []
# for case_name, filt in case_filters.items():
#     mask = (
#         (df["log_name"] == filt["source"])
#         & (df["TCIPC_ZeroYawDeflection"] == filt["TCIPC_ZeroYawDeflection"])
#         & (df["TCIPC_TowerClearanceReference"] == filt["TCIPC_TowerClearanceReference"])
#     )
#     df_case = df[mask].copy()
#     df_case["log_name"] = case_name
#     filtered_dfs.append(df_case)
#     print(f"  {case_name}: {len(df_case)} rows")

# df = pd.concat(filtered_dfs, ignore_index=True).sort_values("aep", ignore_index=True)
# df["log_name"] = pd.Categorical(df["log_name"], categories=LOG_NAMES, ordered=True)
# print(f"\nFinal dataframe shape: {df.shape}")

# df.head()
df_baseline = df[
    (df["log_name"] == "Free yaw") & (df["TCIPC_TowerClearanceReference"] == 5.0)
].copy()
df_baseline["log_name"] = "Baseline"
# Expand the baseline so it spans the same design-variable grid as the free-yaw case.
baseline_reference = df_baseline["TCIPC_TowerClearanceReference"].iloc[0]
reference_sum = (
    df_baseline["TCIPC_TowerClearanceReference"].iloc[0]
    + df_baseline["TCIPC_MaxTipDeflection"].iloc[0]
)

baseline_parts = [df_baseline]
tcipc_references = np.sort(
    df.loc[df["log_name"] == "Free yaw", "TCIPC_TowerClearanceReference"].unique()
)

for tcipc_reference in tcipc_references:
    if tcipc_reference == baseline_reference:
        continue

    df_baseline_copy = df_baseline.copy()
    df_baseline_copy["TCIPC_TowerClearanceReference"] = tcipc_reference
    df_baseline_copy["TCIPC_MaxTipDeflection"] = reference_sum - tcipc_reference
    baseline_parts.append(df_baseline_copy)

df_baseline = pd.concat(baseline_parts, ignore_index=True)

LOG_NAMES = [BASELINE_LOG] + [
    name for name in df["log_name"].unique() if name != BASELINE_LOG
]
df = pd.concat([df, df_baseline], ignore_index=True).sort_values(
    "aep", ignore_index=True
)
df

In [ ]:
# import gc
# import sys

# # These variables are only used during df construction and can be freed to save RAM
# _to_delete = [
#     "all_data_dicts",
#     "all_outputs",
#     "all_dfs",
#     "filtered_dfs",
#     "case_filters",
#     "logs_to_load",
# ]

# for name in _to_delete:
#     obj = globals().get(name)
#     if obj is not None:
#         print(f"  del {name:20s}  ({sys.getsizeof(obj):>10,} bytes shallow)")
#         del globals()[name]

# del _to_delete
# gc.collect()
# print("Done.")

## Data exploration

In [ ]:
# Plot the distribution of design variables for each case.
plt.figure()
sns.scatterplot(
    data=df,
    x="TCIPC_TowerClearanceReference",
    y="max_thrust_percent",
    style="log_name",
)
plt.show()

In [ ]:
# Plot several outputs as a function of the design variables.
outputs = [
    "aep",
    "eff_tower_clearance",
    "max_eff_TwrBsMyt",
    "DEL_TwrBsMyt",
    "damage_tower_base",
    "max_eff_RootMyb",
    "DEL_RootMyb",
    "hub_Mxyz",
    "max_pitch_rate_sim",
    "avg_pitch_travel",
]

fig, axs = plt.subplots(len(outputs), len(LOG_NAMES), figsize=(10, 30))

for i, output in enumerate(outputs):
    # Shared colour limits across all log_names for this row.
    all_vals = pd.concat([df[df["log_name"] == ln][output] for ln in LOG_NAMES])
    vmin, vmax = all_vals.min(), all_vals.max()

    for j, log_name in enumerate(LOG_NAMES):
        mask = df["log_name"] == log_name
        scatter = axs[i, j].scatter(
            df[mask]["TCIPC_TowerClearanceReference"],
            df[mask]["max_thrust_percent"],
            c=df[mask][output],
            vmin=vmin,
            vmax=vmax,
        )

        axs[i, j].set_ylabel(labels["max_thrust_percent"])
        axs[i, j].set_xlabel(labels["TCIPC_TowerClearanceReference"])
        axs[i, j].set_ylim((55, 105))
        plt.colorbar(scatter, ax=axs[i, j])

        if i == 0:
            axs[i, j].set_title(log_name)
        if j == 0:
            axs[i, j].annotate(
                labels[output],
                xy=(0, 0.5),
                xytext=(-axs[i, j].yaxis.labelpad - 5, 0),
                xycoords=axs[i, j].yaxis.label,
                textcoords="offset points",
                ha="right",
                va="center",
                rotation=90,
            )

plt.show()

In [ ]:
# Get an idea of the trade-off between AEP and tower clearance.
fig, ax = plt.subplots()

sns.lineplot(
    data=df[df["TCIPC_MaxTipDeflection"] == 0.0],
    y="aep",
    x="eff_tower_clearance",
    hue="log_name",
)
plt.show()

## Data interpolation

In [ ]:
# Build 2D interpolators and evaluate on a regular grid for contour plotting.
ps_min, ps_max = df["max_thrust_percent"].min(), df["max_thrust_percent"].max()
tip_min, tip_max = (
    df["TCIPC_TowerClearanceReference"].min(),
    df["TCIPC_TowerClearanceReference"].max(),
)

n_points = 50
ps_grid = np.linspace(ps_min, ps_max, n_points)
tip_grid = np.linspace(tip_min, tip_max, n_points)
tcipc_reference_grid, ps_percent_grid = np.meshgrid(tip_grid, ps_grid)

interpolated_data = {}

for log_name in LOG_NAMES:
    interpolated_data[log_name] = {}
    df_log = df[df["log_name"] == log_name]

    points = df_log[list(DESIGN_VARS)].values

    for output in outputs:
        values = df_log[output].values

        grid_values = griddata(
            points,
            values,
            (tcipc_reference_grid, ps_percent_grid),
            method="linear",
        )

        interpolated_data[log_name][output] = grid_values

print(f"Interpolated {len(outputs)} outputs for {len(LOG_NAMES)} datasets")
print(f"Grid shape: {ps_percent_grid.shape}")

In [ ]:
# Plot interpolated contour surfaces for each output and dataset.
fig, axs = plt.subplots(len(outputs), len(LOG_NAMES), figsize=(10, 30))

for i, output in enumerate(outputs):
    # Compute shared color limits across all datasets for this output.
    all_vals = np.concatenate(
        [interpolated_data[ln][output].ravel() for ln in LOG_NAMES]
    )
    vmin = np.nanmin(all_vals)
    vmax = np.nanmax(all_vals)
    if vmin == vmax:
        vmax += 0.001
    levels = np.linspace(vmin, vmax, 9)

    # Reference value from the baseline at the standard operating point.
    ref_val = griddata(
        df[df["log_name"] == BASELINE_LOG][list(DESIGN_VARS)].values,
        df[df["log_name"] == BASELINE_LOG][output].values,
        [REFERENCE_POINT],
        method="linear",
    )[0]

    for j, log_name in enumerate(LOG_NAMES):
        contour = axs[i, j].contourf(
            tcipc_reference_grid,
            ps_percent_grid,
            interpolated_data[log_name][output],
            levels=levels,
            vmin=vmin,
            vmax=vmax,
        )

        # Iso line at the reference value.
        axs[i, j].contour(
            tcipc_reference_grid,
            ps_percent_grid,
            interpolated_data[log_name][output],
            levels=[ref_val],
            colors="k",
            linewidths=0.8,
        )

        axs[i, j].set_ylabel(labels["max_thrust_percent"])
        axs[i, j].set_xlabel(labels["TCIPC_TowerClearanceReference"])
        plt.colorbar(contour, ax=axs[i, j])

        if i == 0:
            axs[i, j].set_title(log_name)
        if j == 0:
            axs[i, j].annotate(
                labels[output],
                xy=(0, 0.5),
                xytext=(-axs[i, j].yaxis.labelpad - 5, 0),
                xycoords=axs[i, j].yaxis.label,
                textcoords="offset points",
                ha="right",
                va="center",
                rotation=90,
            )

In [ ]:
# Investigate one plot in detail.
fig, ax = plt.subplots()

scatter = ax.contourf(
    tcipc_reference_grid,
    ps_percent_grid,
    interpolated_data[LOG_NAMES[1]]["DEL_RootMyb"],
    levels=np.linspace(18, 25, 100),
)

plt.colorbar(scatter)

## Optimization

In [ ]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.unsga3 import UNSGA3
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.ctaea import CTAEA
from pymoo.algorithms.moo.sms import SMSEMOA
from pymoo.algorithms.moo.age2 import AGEMOEA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.indicators.hv import Hypervolume
from scipy.interpolate import (
    CloughTocher2DInterpolator,
    LinearNDInterpolator,
    RBFInterpolator,
)
from copy import deepcopy
from itertools import cycle

In [ ]:
class InterpolatorSet:
    """Builds interpolators for all numeric columns of a dataframe subset.

    Given a dataframe (typically filtered to one log_name), this creates a
    scipy interpolator for each numeric column, using the specified design
    variable columns as inputs. Works for any number of dimensions.
    """

    def __init__(self, df, design_vars, method="linear"):
        xy = df[list(design_vars)].values
        self._ndim = len(design_vars)

        if method == "cubic" and self._ndim > 2:
            # CloughTocher only works for 2D; use RBF for higher dimensions.
            self._interpolators = {}
            for col in df.select_dtypes(include=[np.number]).columns:
                if col in design_vars:
                    continue
                rbf = RBFInterpolator(xy, df[col].values, kernel="thin_plate_spline")
                self._interpolators[col] = self._wrap_rbf(rbf)
        else:
            Interpolator = (
                CloughTocher2DInterpolator
                if method == "cubic"
                else LinearNDInterpolator
            )
            self._interpolators = {}
            for col in df.select_dtypes(include=[np.number]).columns:
                if col in design_vars:
                    continue
                self._interpolators[col] = Interpolator(
                    xy, df[col].values, fill_value=0.0
                )

    @staticmethod
    def _wrap_rbf(rbf):
        """Wrap RBFInterpolator to accept *args like LinearNDInterpolator."""

        def wrapper(*args):
            xi = np.column_stack([np.atleast_1d(a) for a in args])
            result = rbf(xi)
            return result[0] if result.size == 1 else result

        return wrapper

    def __getitem__(self, column):
        return self._interpolators[column]

In [ ]:
class OptimizationProblem:
    """Declarative specification of a multi-objective optimization problem.

    Objectives are specified as dicts:
        {"variable": str, "direction": "minimize" | "maximize", "label": str}

    Constraints are specified as dicts:
        {
            "name": str,
            "variable": str,
            "label": str,
            "threshold": float,
            "direction": "below" | "above"
        }
    """

    def __init__(self, objectives, constraints, xl, xu):
        self.objectives = objectives
        self.constraints = constraints
        self.xl = np.array(xl)
        self.xu = np.array(xu)

    def to_pymoo(self, interp_set):
        """Build a pymoo ElementwiseProblem from this specification.

        Pymoo always minimizes, so objectives with direction="maximize" are
        negated internally. Works for any number of design variables.
        """
        objectives = self.objectives
        constraints = self.constraints
        xl = self.xl
        xu = self.xu
        n_var = len(xl)

        # Negate "maximize" objectives so pymoo can minimize them all.
        obj_callables = []
        for obj in objectives:
            interp = interp_set[obj["variable"]]
            sign = -1.0 if obj["direction"] == "maximize" else 1.0
            obj_callables.append(lambda x, _i=interp, _s=sign: _s * _i(*x))

        # Build constraint callables (G <= 0 is feasible).
        con_callables = []
        for con in constraints:
            interp = interp_set[con["variable"]]
            threshold = con["threshold"]
            if con["direction"] == "below":
                con_callables.append(lambda x, _i=interp, _t=threshold: _i(*x) - _t)
            else:
                con_callables.append(lambda x, _i=interp, _t=threshold: _t - _i(*x))

        class _Problem(ElementwiseProblem):
            def __init__(self):
                super().__init__(
                    n_var=n_var,
                    n_obj=len(objectives),
                    n_ieq_constr=len(constraints),
                    xl=xl,
                    xu=xu,
                )

            def _evaluate(self, x, out, *args, **kwargs):
                out["F"] = [f(x) for f in obj_callables]
                out["G"] = [g(x) for g in con_callables]

        return _Problem()

In [ ]:
class OptimizationStudy:
    """Runs multi-objective optimization across datasets and algorithms.

    Uses InterpolatorSets to evaluate a surrogate-based optimization problem
    for each log_name in the dataframe. Results are returned as DataFrames
    suitable for plotting with seaborn.
    """

    def __init__(
        self,
        df,
        problem,
        algorithms,
        termination,
        design_vars,
        interp_method="linear",
        seed=1,
        verbose=False,
    ):
        self.df = df
        self.problem = problem
        self.algorithms = algorithms
        self.termination = termination
        self.design_vars = design_vars
        self.interp_method = interp_method
        self.seed = seed
        self.verbose = verbose

        self.log_names = sorted(list(dict.fromkeys(df["log_name"])))
        self.results = {}
        self.interp_sets = {}

    def run(self):
        """Run all optimizations for each (log_name, algorithm) pair."""
        for log_name in self.log_names:
            df_log = self.df[self.df["log_name"] == log_name]
            self.interp_sets[log_name] = InterpolatorSet(
                df_log, self.design_vars, method=self.interp_method
            )

        for log_name in self.log_names:
            interp_set = self.interp_sets[log_name]
            for algo_name, algo_factory in self.algorithms.items():
                result = minimize(
                    self.problem.to_pymoo(interp_set),
                    deepcopy(algo_factory),
                    deepcopy(self.termination),
                    seed=self.seed,
                    save_history=True,
                    verbose=self.verbose,
                )
                self.results[(log_name, algo_name)] = result
                if result.F is None:
                    print("Didn't get any results...")
                else:
                    print(f"  Done: {log_name} -> {len(result.F)} solutions")

    def _calculate_hv_history(self, result, ref_point=None):
        """Compute hypervolume at each generation from a pymoo result."""
        hist_F = []
        for algo in result.history:
            opt = algo.opt
            feas = np.where(opt.get("feasible"))[0]
            hist_F.append(opt.get("F")[feas])

        if ref_point is not None:
            signs = np.array(
                [
                    -1.0 if o["direction"] == "maximize" else 1.0
                    for o in self.problem.objectives
                ]
            )
            pymoo_ref = np.array(ref_point) * signs
        else:
            pymoo_ref = result.F.max(axis=0)

        metric = Hypervolume(
            ref_point=pymoo_ref,
            norm_ref_point=False,
            zero_to_one=False,
        )
        return [metric.do(_F) for _F in hist_F]

    def convergence_to_dataframe(self, ref_point=None, normalize=False):
        """Return hypervolume convergence history as a DataFrame.

        Each row is one generation for one (log_name, algorithm) pair.
        """
        rows = []
        for (log_name, algo_name), result in self.results.items():
            hv = np.array(self._calculate_hv_history(result, ref_point=ref_point))
            if normalize:
                hv_min, hv_max = hv[0], hv[-1]
                if hv_max > hv_min:
                    hv = (hv - hv_min) / (hv_max - hv_min)
            for gen, h in enumerate(hv):
                rows.append(
                    {
                        "log_name": log_name,
                        "algo_name": algo_name,
                        "generation": gen,
                        "hypervolume": h,
                    }
                )
        return pd.DataFrame(rows)

    def to_dataframe(self):
        """Return all Pareto-optimal solutions as a seaborn-friendly DataFrame.

        Each row is one Pareto-optimal solution. Columns include the design
        variables, objective values (in original space, not pymoo-negated),
        constraint actual values, and all other interpolated outputs.
        """
        signs = np.array(
            [
                -1.0 if o["direction"] == "maximize" else 1.0
                for o in self.problem.objectives
            ]
        )

        rows = []
        for (log_name, algo_name), result in self.results.items():
            interp_set = self.interp_sets[log_name]
            for i in range(len(result.X)):
                x = result.X[i]
                row = {
                    "log_name": log_name,
                    "algo_name": algo_name,
                }

                # Design variables.
                for j, dv in enumerate(self.design_vars):
                    row[dv] = x[j]

                # Objectives in the original space.
                F_original = result.F[i] * signs
                for j, obj in enumerate(self.problem.objectives):
                    row[obj["variable"]] = F_original[j]

                # Constraint actual values.
                for j, con in enumerate(self.problem.constraints):
                    if con["direction"] == "below":
                        row[con["variable"]] = result.G[i, j] + con["threshold"]
                    else:
                        row[con["variable"]] = con["threshold"] - result.G[i, j]

                # All other interpolated output variables.
                for col in interp_set._interpolators:
                    if col not in row:
                        row[col] = float(interp_set[col](*x))

                rows.append(row)

        return pd.DataFrame(rows)

In [ ]:
# Compute constraint thresholds at the baseline reference operating point.
df_baseline = df[df["log_name"] == BASELINE_LOG]
reference_interp = InterpolatorSet(df_baseline, DESIGN_VARS, method="linear")

reference_variables = [
    "eff_tower_clearance",
    "aep",
    "max_eff_TwrBsMyt",
    "DEL_TwrBsMyt",
    "damage_tower_base",
    "max_eff_RootMyb",
    "DEL_RootMyb",
    "hub_Mxyz",
    "avg_pitch_travel",
]

ref_values = {}
for var in reference_variables:
    ref_values[var] = float(reference_interp[var](*REFERENCE_POINT))

print(f"Reference settings ({BASELINE_LOG} at {REFERENCE_POINT}):")
for var, val in ref_values.items():
    print(f"  {var:20s} = {val:.4g} {labels[var].split('(')[-1].rstrip(')')}")

In [ ]:
# Define the optimization problem with 2D design space.
# All structural loads are constrained to not exceed the reference settings.
problem = OptimizationProblem(
    objectives=[
        {
            "variable": "eff_tower_clearance",
            "direction": "maximize",
            "label": "Tower clearance (m)",
        },
        {
            "variable": "aep",
            "direction": "maximize",
            "label": "AEP (GWh)",
        },
    ],
    constraints=[
        {
            "name": "Max tower base Myt",
            "variable": "max_eff_TwrBsMyt",
            "label": "Max tower base Myt (MNm)",
            "threshold": ref_values["max_eff_TwrBsMyt"] * 1.001,
            "direction": "below",
        },
        {
            "name": "DEL tower base Myt",
            "variable": "DEL_TwrBsMyt",
            "label": "DEL tower base Myt (MNm)",
            "threshold": ref_values["DEL_TwrBsMyt"] * 1.001,
            "direction": "below",
        },
        {
            "name": "Max root Myb",
            "variable": "max_eff_RootMyb",
            "label": "Max root Myb (MNm)",
            "threshold": ref_values["max_eff_RootMyb"] * 1.001,
            "direction": "below",
        },
        {
            "name": "DEL root Myb",
            "variable": "DEL_RootMyb",
            "label": "DEL root Myb (MNm)",
            "threshold": ref_values["DEL_RootMyb"] * 1.001,
            "direction": "below",
        },
    ],
    xl=[tip_min, ps_min],
    xu=[tip_max, ps_max],
)

# Specify the algorithms to compare.
ref_dirs = get_reference_directions("uniform", 2, n_partitions=99)

algorithms = {
    "NSGA2": NSGA2(pop_size=100),
}

termination = get_termination("n_gen", 50)

# Run the study across all datasets and algorithms.
study = OptimizationStudy(
    df,
    problem=problem,
    algorithms=algorithms,
    termination=termination,
    design_vars=DESIGN_VARS,
    interp_method="cubic",
)
study.run()

In [ ]:
# Hypervolume convergence comparison across all runs.
df_conv = study.convergence_to_dataframe(ref_point=(0, 0), normalize=False)

fig, ax = plt.subplots()
sns.lineplot(data=df_conv, x="generation", y="hypervolume", hue="log_name", ax=ax)
plt.show()

In [ ]:
# Pareto front comparison across all runs.
df_pareto = study.to_dataframe()

fig, ax = plt.subplots()
sns.scatterplot(data=df_pareto, y="aep", x="eff_tower_clearance", hue="log_name", ax=ax)
plt.show()

In [ ]:
# Design space for Pareto-optimal solutions.
df_pareto = study.to_dataframe()

# For the baseline, TCIPC_TowerClearanceReference has no effect so we set it to zero.
df_pareto.loc[
    df_pareto["log_name"] == BASELINE_LOG, "TCIPC_TowerClearanceReference"
] = 0.0

fig, ax = plt.subplots()
sns.scatterplot(
    data=df_pareto,
    y="max_thrust_percent",
    x="TCIPC_TowerClearanceReference",
    hue="log_name",
    ax=ax,
)
ax.set_ylabel(labels["max_thrust_percent"])
ax.set_xlabel(labels["TCIPC_TowerClearanceReference"])
ax.legend(loc="upper left")
plt.show()

In [ ]:
# Constraint satisfaction for Pareto-optimal solutions.
df_pareto = study.to_dataframe()

constraint_cols = [c["variable"] for c in problem.constraints]

fig, axs = plt.subplots(1, len(constraint_cols), squeeze=False, figsize=(10, 4))

for j, con in enumerate(problem.constraints):
    sns.scatterplot(
        data=df_pareto,
        x="aep",
        y=con["variable"],
        hue="log_name",
        ax=axs[0, j],
    )
    axs[0, j].axhline(con["threshold"], color="red", linestyle="--", label="Threshold")
    axs[0, j].set_title(con["name"])
    axs[0, j].set_ylabel(con["label"])
    axs[0, j].get_legend().remove()

plt.tight_layout()
plt.show()

In [ ]:
# Trade-off between pitch actuation and blade root fatigue for Pareto solutions.
df_pareto = study.to_dataframe()

fig, ax = plt.subplots()
sns.scatterplot(
    data=df_pareto,
    x="avg_pitch_travel",
    y="DEL_RootMyb",
    hue="log_name",
    ax=ax,
)
ax.set_xlabel(labels["avg_pitch_travel"])
ax.set_ylabel(labels["DEL_RootMyb"])
ax.legend(loc="lower right")
plt.show()

In [ ]:
# Average pitch travel for Pareto-optimal solutions compared to the reference.
df_pareto = study.to_dataframe()
ref_avg_pitch_travel = float(reference_interp["avg_pitch_travel"](*REFERENCE_POINT))

fig, ax = plt.subplots()
sns.scatterplot(
    data=df_pareto,
    y="aep",
    x="avg_pitch_travel",
    hue="log_name",
    ax=ax,
)
ax.axvline(ref_avg_pitch_travel, color="red", linestyle="--", label="Reference")
ax.set_xlabel(labels["avg_pitch_travel"])
ax.legend()
plt.show()

## Journal plots

In [ ]:
from pathlib import Path
from cmcrameri import cm

# Apply the journal style.
plt.style.use("journal.mplstyle")

FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Retrieve default figure dimensions from the style sheet.
default_width, default_height = plt.rcParams["figure.figsize"]

In [ ]:
# Contour plot: AEP and tower clearance as rows, one column per dataset.
# journal_constraints = {
#     "max_eff_TwrBsMyt": {"vmin": 250, "vmax": 390, "n_levels": 13, "n_cbar_ticks": 5},
#     "DEL_TwrBsMyt": {"vmin": 45, "vmax": 54, "n_levels": 13, "n_cbar_ticks": 5},
#     "max_eff_RootMyb": {"vmin": 68, "vmax": 108, "n_levels": 13, "n_cbar_ticks": 5},
#     "DEL_RootMyb": {"vmin": 18, "vmax": 37, "n_levels": 13, "n_cbar_ticks": 5},
# }
journal_constraints = {
    "max_eff_TwrBsMyt": {"vmin": 310, "vmax": 422, "n_levels": 13, "n_cbar_ticks": 5},
    "DEL_TwrBsMyt": {"vmin": 46, "vmax": 54, "n_levels": 13, "n_cbar_ticks": 5},
    "max_eff_RootMyb": {"vmin": 80, "vmax": 124, "n_levels": 13, "n_cbar_ticks": 5},
    "DEL_RootMyb": {"vmin": 18, "vmax": 34, "n_levels": 13, "n_cbar_ticks": 5},
}

fig, axs = plt.subplots(
    len(journal_constraints),
    len(LOG_NAMES),
    sharex=True,
    sharey=True,
    figsize=(default_width / 1.2, 2 * default_height),
)

for i, (output, cfg) in enumerate(journal_constraints.items()):
    vmin = cfg["vmin"]
    vmax = cfg["vmax"]
    levels = np.linspace(vmin, vmax, cfg["n_levels"])

    # Reference value from the baseline at the standard operating point.
    ref_val = griddata(
        df[df["log_name"] == BASELINE_LOG][list(DESIGN_VARS)].values,
        df[df["log_name"] == BASELINE_LOG][output].values,
        [REFERENCE_POINT],
        method="linear",
    )[0]

    for j, log_name in enumerate(LOG_NAMES):
        contour = axs[i, j].contourf(
            tcipc_reference_grid,
            ps_percent_grid,
            interpolated_data[log_name][output],
            levels=levels,
            vmin=vmin,
            vmax=vmax,
            # extend="both",  # Danger! Adjust vmin and vmax before enabling this.
        )

        # Iso line at the reference value.
        cs = axs[i, j].contour(
            tcipc_reference_grid,
            ps_percent_grid,
            interpolated_data[log_name][output],
            levels=[ref_val],
            colors="k",
            linewidths=1.0,
            linestyles="dashed",
        )

        # Only add the colourbar on the rightmost column.
        if j == len(LOG_NAMES) - 1:
            cbar = plt.colorbar(
                contour,
                ax=axs[i, j],
                format="%.0f",
                ticks=np.linspace(vmin, vmax, cfg["n_cbar_ticks"]),
            )
            cbar.set_label(labels[output])  # , rotation=0, horizontalalignment="left")

        if i == 0:
            axs[i, j].set_title(log_name)

        axs[i, j].set_yticks(np.arange(60, 101, 10))
        # axs[i, j].tick_params("x", top=True)

# Add a legend for the reference iso line in the top-right subplot.
ref_line = plt.Line2D([], [], color="k", linestyle="dashed", linewidth=1.0)
axs[-1, -1].legend([ref_line], ["Reference"], loc="lower right")

# Shared axis labels on the outer edges only.
for ax in axs[-1, :]:
    ax.set_xlabel(labels["TCIPC_TowerClearanceReference"])
for ax in axs[:, 0]:
    ax.set_ylabel(labels["max_thrust_percent"])

fig.savefig(FIGURE_DIR / "contour_constraints.pdf")
plt.show()

In [ ]:
# Contour plot: AEP and tower clearance as rows, one column per dataset.
journal_outputs = {
    "aep": {"vmin": 74, "vmax": 86, "n_levels": 13, "n_cbar_ticks": 5},
    "eff_tower_clearance": {"vmin": 10, "vmax": 20, "n_levels": 13, "n_cbar_ticks": 5},
}

fig, axs = plt.subplots(
    len(journal_outputs),
    len(LOG_NAMES),
    sharex=True,
    sharey=True,
    figsize=(default_width / 1.2, 1.0 * default_height),
)

for i, (output, cfg) in enumerate(journal_outputs.items()):
    vmin = cfg["vmin"]
    vmax = cfg["vmax"]
    levels = np.linspace(vmin, vmax, cfg["n_levels"])

    # Reference value from the baseline at the standard operating point.
    ref_val = griddata(
        df[df["log_name"] == BASELINE_LOG][list(DESIGN_VARS)].values,
        df[df["log_name"] == BASELINE_LOG][output].values,
        [REFERENCE_POINT],
        method="cubic",
    )[0]

    for j, log_name in enumerate(LOG_NAMES):
        contour = axs[i, j].contourf(
            tcipc_reference_grid,
            ps_percent_grid,
            interpolated_data[log_name][output],
            levels=levels,
            vmin=vmin,
            vmax=vmax,
            # extend="both",  # Danger! Adjust vmin and vmax before enabling this.
        )

        # Iso line at the reference value.
        cs = axs[i, j].contour(
            tcipc_reference_grid,
            ps_percent_grid,
            interpolated_data[log_name][output],
            levels=[ref_val],
            colors="k",
            linewidths=1.0,
            linestyles="dashed",
        )

        # Only add the colourbar on the rightmost column.
        if j == len(LOG_NAMES) - 1:
            cbar = plt.colorbar(
                contour,
                ax=axs[i, j],
                format="%.0f",
                ticks=np.linspace(vmin, vmax, cfg["n_cbar_ticks"]),
            )
            cbar.set_label(labels[output])

        if i == 0:
            axs[i, j].set_title(log_name)

        axs[i, j].set_yticks(np.arange(60, 101, 10))
        # axs[i, j].tick_params("x", top=True)

# Add a legend for the reference iso line in the top-right subplot.
ref_line = plt.Line2D([], [], color="k", linestyle="dashed", linewidth=1.0)
axs[-1, -1].legend([ref_line], ["Reference"], loc="lower right")

# Shared axis labels on the outer edges only.
for ax in axs[-1, :]:
    ax.set_xlabel(labels["TCIPC_TowerClearanceReference"])
for ax in axs[:, 0]:
    ax.set_ylabel(labels["max_thrust_percent"])

fig.savefig(FIGURE_DIR / "contour_aep_clearance.pdf")
plt.show()

### Unconstrained optimization

In [ ]:
# Unconstrained problem: same objectives, no structural-load constraints.
problem_unconstrained = OptimizationProblem(
    objectives=[
        {
            "variable": "eff_tower_clearance",
            "direction": "maximize",
            "label": "Tower clearance (m)",
        },
        {
            "variable": "aep",
            "direction": "maximize",
            "label": "AEP (GWh)",
        },
    ],
    constraints=[],
    xl=[tip_min, ps_min],
    xu=[tip_max, ps_max],
)

study_unconstrained = OptimizationStudy(
    df,
    problem=problem_unconstrained,
    algorithms={"NSGA2": NSGA2(pop_size=500)},
    termination=get_termination("n_gen", 100),
    design_vars=DESIGN_VARS,
    interp_method="cubic",
)
study_unconstrained.run()

In [ ]:
df_unc = study_unconstrained.convergence_to_dataframe(ref_point=(0, 0), normalize=True)

fig, ax = plt.subplots(figsize=(default_width, 0.7 * default_height))
sns.lineplot(
    data=df_unc, x="generation", y="hypervolume", hue="log_name", ax=ax, linewidth=2
)
ax.set_xlabel("Generation (-)")
ax.set_ylabel("Normalized hypervolume (-)")
ax.legend(loc="upper left", bbox_to_anchor=(1.00, 1.045))
fig.savefig(FIGURE_DIR / "unconstrained_convergence.pdf")
plt.show()

In [ ]:
# # Pareto front and design space for unconstrained optimization.
df_unc = study_unconstrained.to_dataframe()

# For the baseline, TCIPC_TowerClearanceReference has no effect so we set it to zero.
df_unc.loc[df_unc["log_name"] == BASELINE_LOG, "TCIPC_TowerClearanceReference"] = 0.0

# fig, axs = plt.subplots(1, 2, figsize=(default_width, 0.7 * default_height))

# # Pareto front.
# sns.scatterplot(
#     data=df_unc,
#     x="aep",
#     y="eff_tower_clearance",
#     hue="log_name",
#     ax=axs[0],
#     legend=False,
#     edgecolor="none",
#     s=20,
# )

# # Design space.
# sns.scatterplot(
#     data=df_unc,
#     x="ps_percent",
#     y="TCIPC_TowerClearanceReference",
#     hue="log_name",
#     ax=axs[1],
#     edgecolor="none",
#     s=20,
# )

# axs[0].set_xlabel(labels["aep"])
# axs[0].set_ylabel(labels["eff_tower_clearance"])
# axs[1].set_xlabel(labels["ps_percent"])
# axs[1].set_ylabel(labels["TCIPC_TowerClearanceReference"])

# axs[0].set_xlim((70, 90))
# axs[0].set_ylim((9.0, 24))
# axs[1].set_xlim((0.48, 1.02))
# axs[1].set_ylim((-0.8, 20.0))
# axs[1].set_xticks(np.arange(0.6, 1.01, 0.2))

# # Place legend outside to the right, aligned with the top of the axes.
# handles, legend_labels = axs[1].get_legend_handles_labels()
# axs[1].legend(handles, legend_labels, loc="upper left", bbox_to_anchor=(1.00, 1.045))

# fig.savefig(FIGURE_DIR / "unconstrained_pareto_design.pdf")
# plt.show()

In [ ]:
# Unconstrained Pareto front as a standalone figure.
fig, ax = plt.subplots(figsize=(0.4 * default_width, 0.7 * default_height))

sns.scatterplot(
    data=df_unc,
    y="aep",
    x="eff_tower_clearance",
    hue="log_name",
    ax=ax,
    legend=False,
    edgecolor="none",
    s=12,
)

ax.plot(ref_values["eff_tower_clearance"], ref_values["aep"], "x", markersize=8)

ax.set_ylabel(labels["aep"])
ax.set_xlabel(labels["eff_tower_clearance"])
ax.set_ylim((75.5, 86.0))
ax.set_xlim((10.0, 20.0))

fig.savefig(FIGURE_DIR / "unconstrained_pareto.pdf")
plt.show()


In [ ]:
# Unconstrained design space as a standalone figure (wider to accommodate legend).
fig, ax = plt.subplots(figsize=(0.6 * default_width, 0.7 * default_height))

ax.plot(0, 80, "x", markersize=8, label="Reference")

sns.scatterplot(
    data=df_unc,
    y="max_thrust_percent",
    x="TCIPC_TowerClearanceReference",
    hue="log_name",
    ax=ax,
    edgecolor="none",
    s=12,
)


ax.set_ylabel(labels["max_thrust_percent"])
ax.set_xlabel(labels["TCIPC_TowerClearanceReference"])
ax.set_ylim((58, 102))
ax.set_xlim((-0.8, 30.8))
ax.set_yticks(np.arange(60, 101, 10))

handles, legend_labels = ax.get_legend_handles_labels()
ax.legend(handles, legend_labels, loc="upper left", bbox_to_anchor=(1.00, 1.045))

fig.savefig(FIGURE_DIR / "unconstrained_design.pdf")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(0.48 * default_width, 0.7 * default_height))

ax.plot(80, ref_values["avg_pitch_travel"], "x", markersize=8, label="Reference")

sns.scatterplot(
    df_unc,
    y="avg_pitch_travel",
    x="max_thrust_percent",
    hue="log_name",
    ax=ax,
    edgecolor="none",
    s=12,
)

ax.set_ylabel("Actuator duty cycle (-)")
ax.set_xlabel(labels["max_thrust_percent"])
ax.set_ylim((0, 1.0))
handles, legend_labels = ax.get_legend_handles_labels()
ax.legend(handles, legend_labels, loc="upper left", bbox_to_anchor=(1.00, 1.045))

fig.savefig(FIGURE_DIR / "unconstrained_adc.pdf")
plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(0.4 * default_width, 0.7 * default_height))

# sns.boxplot(
#     df_unc,
#     y="avg_pitch_travel",
#     x="log_name",
#     hue="log_name",
#     ax=ax,
#     width=0.6,
#     whis=1e5,  # Ensures outliers are not plotted as separate dots.
# )

# ax.set_ylabel("Actuator duty cycle (-)")
# ax.set_xlabel("")
# ax.set_ylim((0, 1.2))

# # fig.savefig(FIGURE_DIR / "unconstrained_adc.pdf")
# plt.show()

In [ ]:
temp = df_unc.groupby("log_name")["avg_pitch_travel"].mean()
temp / temp["Baseline"]

### Constrained optimization

In [ ]:
# Constrained problem: structural loads + average pitch travel constraint.
problem_constrained = OptimizationProblem(
    objectives=[
        {
            "variable": "eff_tower_clearance",
            "direction": "maximize",
            "label": "Tower clearance (m)",
        },
        {
            "variable": "aep",
            "direction": "maximize",
            "label": "AEP (GWh)",
        },
    ],
    constraints=[
        {
            "name": "Tower ultimate",
            "variable": "max_eff_TwrBsMyt",
            "label": "Tower ultimate (MNm)",
            "threshold": ref_values["max_eff_TwrBsMyt"] * 1.02,
            "direction": "below",
        },
        {
            "name": "Tower fatigue",
            "variable": "DEL_TwrBsMyt",
            "label": "Tower fatigue (MNm)",
            "threshold": ref_values["DEL_TwrBsMyt"] * 1.0,
            "direction": "below",
        },
        {
            "name": "Blade ultimate",
            "variable": "max_eff_RootMyb",
            "label": "Blade ultimate (MNm)",
            "threshold": ref_values["max_eff_RootMyb"] * 1.02,
            "direction": "below",
        },
        {
            "name": "Blade fatigue",
            "variable": "DEL_RootMyb",
            "label": "Blade fatigue (MNm)",
            "threshold": ref_values["DEL_RootMyb"] * 1.0,
            "direction": "below",
        },
        # {
        #     "name": "ADC",
        #     "variable": "avg_pitch_travel",
        #     "label": "Avg pitch travel (deg)",
        #     "threshold": ref_values["avg_pitch_travel"] * 3.0,
        #     "direction": "below",
        # },
    ],
    xl=[tip_min, ps_min],
    xu=[tip_max, ps_max],
)

study_constrained = OptimizationStudy(
    df,
    problem=problem_constrained,
    algorithms={"NSGA2": NSGA2(pop_size=500)},
    termination=get_termination("n_gen", 100),
    design_vars=DESIGN_VARS,
    interp_method="cubic",
)
study_constrained.run()

In [ ]:
df_conv = study_constrained.convergence_to_dataframe(ref_point=(0, 0), normalize=True)

fig, ax = plt.subplots(figsize=(default_width, 0.7 * default_height))
sns.lineplot(
    data=df_conv, x="generation", y="hypervolume", hue="log_name", ax=ax, linewidth=2
)
ax.set_xlabel("Generation (-)")
ax.set_ylabel("Normalized hypervolume (-)")
ax.legend(loc="upper left", bbox_to_anchor=(1.00, 1.045))
fig.savefig(FIGURE_DIR / "constrained_convergence.pdf")
plt.show()

In [ ]:
# # Pareto front and design space for constrained optimization.
df_con = study_constrained.to_dataframe()

# For the baseline, TCIPC_TowerClearanceReference has no effect so we set it to zero.
df_con.loc[df_con["log_name"] == BASELINE_LOG, "TCIPC_TowerClearanceReference"] = 0.0

# fig, axs = plt.subplots(1, 2, figsize=(default_width, 0.7 * default_height))

# # Pareto front.
# sns.scatterplot(
#     data=df_con,
#     x="aep",
#     y="eff_tower_clearance",
#     hue="log_name",
#     ax=axs[0],
#     legend=False,
#     edgecolor="none",
#     s=20,
# )

# # Design space.
# sns.scatterplot(
#     data=df_con,
#     x="ps_percent",
#     y="TCIPC_TowerClearanceReference",
#     hue="log_name",
#     ax=axs[1],
#     edgecolor="none",
#     s=20,
# )
# axs[0].set_xlabel(labels["aep"])
# axs[0].set_ylabel(labels["tower_clearance"])
# axs[1].set_xlabel(labels["ps_percent"])
# axs[1].set_ylabel(labels["TCIPC_TowerClearanceReference"])

# axs[0].set_xlim((70, 90))
# axs[0].set_ylim((9.0, 24))
# axs[1].set_xlim((0.48, 1.02))
# axs[1].set_ylim((-0.8, 20.0))
# axs[1].set_xticks(np.arange(0.6, 1.01, 0.2))

# # Place legend outside to the right, aligned with the top of the axes.
# handles, legend_labels = axs[1].get_legend_handles_labels()
# axs[1].legend(handles, legend_labels, loc="upper left", bbox_to_anchor=(1.00, 1.045))

# fig.savefig(FIGURE_DIR / "constrained_pareto_design.pdf")
# plt.show()

In [ ]:
# Constrained Pareto front as a standalone figure.
fig, ax = plt.subplots(figsize=(0.4 * default_width, 0.7 * default_height))

sns.scatterplot(
    data=df_con,
    y="aep",
    x="eff_tower_clearance",
    hue="log_name",
    ax=ax,
    legend=False,
    edgecolor="none",
    s=12,
)

ax.plot(ref_values["eff_tower_clearance"], ref_values["aep"], "x", markersize=8)

ax.set_ylabel(labels["aep"])
ax.set_xlabel(labels["eff_tower_clearance"])
ax.set_ylim((75.5, 86.0))
ax.set_xlim((10.0, 20.0))

fig.savefig(FIGURE_DIR / "constrained_pareto.pdf")
fig.savefig(FIGURE_DIR / "constrained_pareto.svg")

plt.show()


In [ ]:
# Calculate the slope:
import scipy

In [ ]:
for log_name in df_con["log_name"].unique():
    close_to_reference = df_con[
        (df_con["eff_tower_clearance"] < 13.75) & (df_con["log_name"] == log_name)
    ]
    res = scipy.stats.linregress(
        close_to_reference["eff_tower_clearance"], close_to_reference["aep"]
    )
    print(f"{log_name=}, slope = {res.slope}")

In [ ]:
# Constrained design space as a standalone figure (wider to accommodate legend).
fig, ax = plt.subplots(figsize=(0.6 * default_width, 0.7 * default_height))

ax.plot(0, 80, "x", markersize=8, label="Reference")

sns.scatterplot(
    data=df_con,
    y="max_thrust_percent",
    x="TCIPC_TowerClearanceReference",
    hue="log_name",
    ax=ax,
    edgecolor="none",
    s=12,
)

ax.set_ylabel(labels["max_thrust_percent"])
ax.set_xlabel(labels["TCIPC_TowerClearanceReference"])
ax.set_ylim((58, 102))
ax.set_xlim((-0.8, 30.8))
ax.set_yticks(np.arange(60, 101, 10))

handles, legend_labels = ax.get_legend_handles_labels()
ax.legend(handles, legend_labels, loc="upper left", bbox_to_anchor=(1.00, 1.045))

fig.savefig(FIGURE_DIR / "constrained_design.pdf")
plt.show()


In [ ]:
# Constraint satisfaction for constrained Pareto-optimal solutions.
df_con = study_constrained.to_dataframe()

fig, axs = plt.subplots(
    2,
    2,
    squeeze=False,
    figsize=(default_width, 1.3 * default_height),
)

for i, constraint in enumerate(problem_constrained.constraints):
    row, col = divmod(i, axs.shape[1])
    axs[row, col].axhline(
        constraint["threshold"], color="red", linestyle="--", label="Reference"
    )
    sns.scatterplot(
        data=df_con,
        x="max_thrust_percent",
        y=constraint["variable"],
        hue="log_name",
        ax=axs[row, col],
        edgecolor="none",
        s=12,
    )
    axs[row, col].set_xlabel(labels["max_thrust_percent"])
    axs[row, col].set_title(constraint["name"])
    axs[row, col].set_ylabel(constraint["label"])
    axs[row, col].get_legend().remove()

    # axs[row, col].set_xlim((76.5, 88.5))
    # axs[row, col].set_xticks(np.arange(78, 89, 2))


# Align the top of the legend with the top of the top-right subplot axes.
handles, legend_labels = axs[0, 0].get_legend_handles_labels()
axs[0, -1].legend(
    handles, legend_labels, loc="upper left", bbox_to_anchor=(1.00, 1.045)
)

fig.savefig(FIGURE_DIR / "constrained_constraints.pdf")
# fig.savefig(FIGURE_DIR / "constrained_constraints.svg")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(0.48 * default_width, 0.7 * default_height))

ax.plot(80, ref_values["avg_pitch_travel"], "x", markersize=8, label="Reference")

sns.scatterplot(
    df_con,
    y="avg_pitch_travel",
    x="max_thrust_percent",
    hue="log_name",
    ax=ax,
    edgecolor="none",
    s=12,
)

ax.set_ylabel("Actuator duty cycle (-)")
ax.set_xlabel(labels["max_thrust_percent"])
ax.set_ylim((0, 1.0))
handles, legend_labels = ax.get_legend_handles_labels()
ax.legend(handles, legend_labels, loc="upper left", bbox_to_anchor=(1.00, 1.045))

fig.savefig(FIGURE_DIR / "constrained_adc.pdf")
plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(0.4 * default_width, 0.7 * default_height))

# sns.boxplot(
#     df_con,
#     y="avg_pitch_travel",
#     x="log_name",
#     hue="log_name",
#     ax=ax,
#     width=0.6,
#     whis=1e5,  # Ensures outliers are not plotted as separate dots.
# )

# ax.set_ylabel("Actuator duty cycle (-)")
# ax.set_xlabel("")
# ax.set_ylim((0, 1.2))

# # fig.savefig(FIGURE_DIR / "constrained_adc.pdf")
# plt.show()

In [ ]:
temp = df_con.groupby("log_name")["avg_pitch_travel"].mean()
temp / temp["Baseline"]

## Statistics

In [ ]:
# Quantify improvements relative to the baseline reference operating point.
ref_aep = float(reference_interp["aep"](*REFERENCE_POINT))
ref_clearance = float(reference_interp["eff_tower_clearance"](*REFERENCE_POINT))
print(f"Reference: AEP = {ref_aep:.3f} GWh, tower clearance = {ref_clearance:.3f} m\n")

for study_label, df_pareto in [("Unconstrained", df_unc), ("Constrained", df_con)]:
    print(f"=== {study_label} ===\n")

    for log_name in LOG_NAMES:
        df_case = df_pareto[df_pareto["log_name"] == log_name]

        # Find the Pareto point closest to the reference AEP.
        idx_aep = (df_case["aep"] - ref_aep).abs().idxmin()
        clearance_at_ref_aep = df_case.loc[idx_aep, "eff_tower_clearance"]
        delta_clearance = clearance_at_ref_aep - ref_clearance

        # Find the Pareto point closest to the reference tower clearance.
        idx_clr = (df_case["eff_tower_clearance"] - ref_clearance).abs().idxmin()
        aep_at_ref_clearance = df_case.loc[idx_clr, "aep"]
        delta_aep = aep_at_ref_clearance - ref_aep

        print(f"{log_name}:")
        print(
            f"  At ref AEP ({ref_aep:.2f} GWh): clearance = {clearance_at_ref_aep:.2f} m  ({delta_clearance:+.2f} m, {delta_clearance / ref_clearance:+.1%})"
        )
        print(
            f"  At ref clearance ({ref_clearance:.2f} m): AEP = {aep_at_ref_clearance:.2f} GWh  ({delta_aep:+.2f} GWh, {delta_aep / ref_aep:+.1%})"
        )
        print()
    print()